In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("miadul/animal-image-classification-5-species")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/datasets/miadul/animal-image-classification-5-species


### Step 1: Paths aur Directory Configuration
Sub sy pehle check karein k dataset ka internal folder structure kaisa hai. Keras ka image_dataset_from_directory use karne k liye aap ko data ko train/validation split mein divide karna hoga.

In [11]:
import os
import tensorflow as tf
from tensorflow.keras import layers, models
import warnings
warnings.filterwarnings('ignore')

# Jo path kagglehub ne diya hai usko set karein
# Aam tor par kagglehub data ko subfolders (Cat, Cow, Lion, etc.) mein download karta hai
DATASET_DIR = path 

# Agar path k andar mazeed aik 'Final_Dataset' ya '5-species' ka folder ho to check kar lein:
print(os.listdir(DATASET_DIR))


['animals_dataset']


### Step 2: 
Data Loading & Splitting (Kaggle Server API Optimization)Yahan hum TensorFlow ka standard utility generator use karenge jo bina storage barhaye data ko split bhi kar deta hai aur optimize bhi rakhta hai:

In [12]:
IMG_SIZE = (224, 224) # MobileNetV2 / ResNet50 k liye standard size
BATCH_SIZE = 32

# 1. Training Dataset (80% Data)
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

Found 629 files belonging to 1 classes.
Using 504 files for training.


In [13]:
# 2. Validation Dataset (20% Data)
val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)



Found 629 files belonging to 1 classes.
Using 125 files for validation.


In [14]:
# Classes check karein (Cat, Cow, Lion, Deer, Dog)
class_names = train_ds.class_names
print("Detected Classes:", class_names)

Detected Classes: ['animals_dataset']


### Step 3: Performance Buffering (Autotune)
Kaggle server par background computational speed ko barhane aur bottlenecks se bachne k liye dataset ko cache aur prefetch zaroor karein:

In [15]:
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)


### Step 4: Pre-trained Model Implementation (Transfer Learning)
Chunke yeh Multi-Class Classification hai, hamara output node 5 hoga aur binary crossentropy ki jagah hamara loss function sparse_categorical_crossentropy hoga. Hum MobileNetV2 use karenge kyun k yeh kafi fast aur highly accurate hai:

In [16]:
# 1. Base Model Load karein (Weights ImageNet k use honge)
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False, # Classifier head remove kar dia
    weights='imagenet'
)



In [17]:
# 2. Base Model ki layers ko freeze karein taake pre-trained knowledge zaya na ho
base_model.trainable = False



In [18]:
# # 3. Custom Classifier Head design karein (Multi-class architecture)
# model = models.Sequential([
#     # Inputs scaling (MobileNetV inputs layers expects scaling between - and )
#     layersRescaling(/offset=-), 
    
#     base_model,
#     layersGlobalAveragePoolingD(),
#     layersDense(activation='relu'),
#     layersDropout(), # Overfitting sy bachne k liye
#     layersDense(activation='softmax') #  Classes hain is liye Node  aur Softmax
# ])


# # 3. Custom Classifier Head design karein (Multi-class architecture)
model = tf.keras.Sequential([
    # Inputs scaling (MobileNetV2 inputs layer expects scaling between -1 and 1)
    tf.keras.layers.Rescaling(1./127.5, offset=-1), 
    
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.2), # Overfitting sy bachne k liye
    tf.keras.layers.Dense(5, activation='softmax') # 5 Classes hain is liye Node 5 aur Softmax
])


## Step 5: Model Compile Aur Train Karein

In [19]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy', # Multi-class integers labels k liye best hai
    metrics=['accuracy']
)

# Model Summary check karein architecture samajhne k liye
model.summary()

# Early stopping callback lagayein takay extra epochs par waqt zaya na ho
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# Training execution
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10, # Pre-trained models 5 sy 10 epochs mein behtreen accuracy day detay hain
    callbacks=[early_stop]
)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling_1 (Rescaling)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ ?                      │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,257,984 (8.61 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 2,257,984 (8.61 MB)

Epoch 1/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 23s 869ms/step - accuracy: 0.9583 - loss: 0.1016 - val_accuracy: 1.0000 - val_loss: 3.7098e-07
Epoch 2/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 1.0000 - loss: 4.0446e-08 - val_accuracy: 1.0000 - val_loss: 2.4796e-08
Epoch 3/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 1.0000 - loss: 8.2784e-09 - val_accuracy: 1.0000 - val_loss: 1.3351e-08
Epoch 4/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 1.0000 - loss: 5.6766e-09 - val_accuracy: 1.0000 - val_loss: 9.5367e-09
Epoch 5/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 1.0000 - loss: 4.4940e-09 - val_accuracy: 1.0000 - val_loss: 9.5367e-09
Epoch 6/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 1.0000 - loss: 6.8593e-09 - val_accuracy: 1.0000 - val_loss: 9.5367e-09
Epoch 7/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 1.0000 - loss: 2.4362e-08 - val_accuracy: 1.0000 - val_loss: 9.5367e-09
